In [1]:
import pandas as pd
import requests
from tqdm import tqdm

# =============================
# CONFIG
# =============================
WEATHER_API_KEY = "49f8c6e8ff3b4c938ba95239260604"
INPUT_FILE = "nashik_onion_clean.csv"
OUTPUT_FILE = "nashik_onion_final_2.csv"

# =============================
# LOAD + CLEAN
# =============================
df = pd.read_csv(INPUT_FILE)

df = df.rename(columns={
    "market_name": "Market",
    "variety": "Grade",
    "p_min": "Min Price (Rs./ Qtl)",
    "p_max": "Max Price (Rs./Qtl)",
    "p_modal": "Modal Price (Rs./Qtl)",
    "t": "Date"
})

df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")

df["Market"] = df["Market"].str.upper().str.strip()
df["Grade"] = df["Grade"].str.upper().str.strip()

# =============================
# TIME FEATURES
# =============================
df["Day"] = df["Date"].dt.day
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

# =============================
# COORDINATES
# =============================
coords = {
    "NASIK": (19.9975, 73.7898),
    "LASALGAON": (20.1420, 74.2390),
    "PIMPALGAON": (20.2000, 74.0000),
    "CHANDVAD": (20.3300, 74.2500),
    "MANMAD": (20.2500, 74.4500),
    "SINNER": (19.8500, 74.0000),
    "DEVALA": (20.4000, 74.5000),
    "LASALGAON(NIPHAD)": (20.1420, 74.2390),
    "PIMPALGAON BASWANT(SAYKHEDA)": (20.1300, 73.9000),
    "LASALGAON(VINCHUR)": (20.1500, 74.2000),
    "NAMPUR": (20.4700, 74.2000),
    "YEOLA": (20.0500, 74.5000),
    "UMRANE": (20.3000, 74.2000),
    "SATANA": (20.6000, 74.0000),
    "KALVAN": (20.5000, 74.0000)
}

# =============================
# CACHE DICTS 🔥
# =============================
weather_cache = {}
festival_cache = {}

# =============================
# FUNCTIONS
# =============================

def get_festival(date):
    if date in festival_cache:
        return festival_cache[date]

    try:
        url = f"https://calender-api-production.up.railway.app/calendar?date={date_str}"
        res = requests.get(url).json()

        fest_list = res.get("state_festivals", {}).get("Maharashtra", [])
        festival = ", ".join(fest_list) if fest_list else "None"

        result = (
            festival,
            int(res.get("is_amavasya", False)),
            int(res.get("is_ekadashi", False)),
            int(res.get("is_purnima", False)),
            res.get("tithi", "Unknown"),
            res.get("nakshatra", "Unknown")
        )

    except:
        result = ("None", 0, 0, 0, "Unknown", "Unknown")

    festival_cache[date] = result
    return result


def get_weather(lat, lon, date):
    key = f"{lat}_{lon}_{date}"

    if key in weather_cache:
        return weather_cache[key]

    try:
        url = "http://api.weatherapi.com/v1/history.json"

        params = {
            "key": WEATHER_API_KEY,
            "q": f"{lat},{lon}",
            "dt": date
        }

        res = requests.get(url, params=params).json()
        day = res["forecast"]["forecastday"][0]["day"]

        temp = day["maxtemp_c"]
        rain = day["totalprecip_mm"]

    except:
        temp, rain = 30, 0

    heat = "Yes" if temp > 35 else "No"

    if rain > 20:
        rain_alert = "Heavy"
    elif rain > 5:
        rain_alert = "Moderate"
    else:
        rain_alert = "No"

    result = (temp, rain, heat, rain_alert)

    weather_cache[key] = result
    return result


def get_agro_advisory(temp, rain, heat, rain_alert):

    if heat == "Yes" and rain < 2:
        return "Irrigation needed | Heat stress risk"
    elif rain_alert == "Heavy":
        return "Heavy rain expected"
    elif rain > 5:
        return "Good moisture"
    else:
        return "Normal conditions"


# =============================
# MAIN LOOP
# =============================
results = []

for i, row in tqdm(df.iterrows(), total=len(df)):

    date = row["Date"].strftime("%Y-%m-%d")
    market = row["Market"]

    lat, lon = coords.get(market, (19.9975, 73.7898))

    # FESTIVAL (cached)
    festival, ama, eka, pur, tithi, nak = get_festival(date)

    # WEATHER (cached)
    temp, rain, heat, rain_alert = get_weather(lat, lon, date)

    advisory = get_agro_advisory(temp, rain, heat, rain_alert)

    results.append({
        "Date": date,
        "Market": market,
        "Grade": row["Grade"],

        "Min Price (Rs./ Qtl)": row["Min Price (Rs./ Qtl)"],
        "Max Price (Rs./Qtl)": row["Max Price (Rs./Qtl)"],
        "Modal Price (Rs./Qtl)": row["Modal Price (Rs./Qtl)"],

        "Festival": festival,
        "is_amavasya": ama,
        "is_ekadashi": eka,
        "is_purnima": pur,
        "Tithi": tithi,
        "Nakshatra": nak,

        "Day": row["Day"],
        "Month": row["Month"],
        "Year": row["Year"],
        "Week": row["Week"],

        "Temp Max (°C)": temp,
        "Rain (mm)": rain,
        "Heat Stress": heat,
        "Rain Alert": rain_alert,

        "Agro Advisory": advisory
    })

# =============================
# SAVE
# =============================
final_df = pd.DataFrame(results)
final_df.to_csv(OUTPUT_FILE, index=False)

print("🔥 DONE FAST!")

100%|██████████| 16337/16337 [1:01:24<00:00,  4.43it/s]


🔥 DONE FAST!


In [3]:
#for fetching caleneder features


import pandas as pd
import requests
from tqdm import tqdm

# =============================
# CONFIG
# =============================
INPUT_FILE = "nashik_onion_clean.csv"
OUTPUT_FILE = "nashik_onion_festival.csv"

# =============================
# LOAD
# =============================
df = pd.read_csv(INPUT_FILE)

df = df.rename(columns={
    "market_name": "Market",
    "variety": "Grade",
    "p_min": "Min Price (Rs./ Qtl)",
    "p_max": "Max Price (Rs./Qtl)",
    "p_modal": "Modal Price (Rs./Qtl)",
    "t": "Date"
})

df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")

df["Market"] = df["Market"].str.upper().str.strip()
df["Grade"] = df["Grade"].str.upper().str.strip()

# =============================
# TIME FEATURES
# =============================
df["Day"] = df["Date"].dt.day
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

# =============================
# FESTIVAL CACHE
# =============================
festival_cache = {}

def get_festival(date):

    if date in festival_cache:
        return festival_cache[date]

    try:
        url = f"https://calender-api-production.up.railway.app/calendar?date={date}"
        res = requests.get(url).json()

        fest_list = res.get("state_festivals", {}).get("Maharashtra", [])
        festival = ", ".join(fest_list) if fest_list else "None"

        result = (
            festival,
            int(res.get("is_amavasya", False)),
            int(res.get("is_ekadashi", False)),
            int(res.get("is_purnima", False)),
            res.get("tithi", "Unknown"),
            res.get("nakshatra", "Unknown")
        )

    except:
        result = ("None", 0, 0, 0, "Unknown", "Unknown")

    festival_cache[date] = result
    return result

# =============================
# LOOP WITH TQDM 🔥
# =============================
results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing Festival Data"):

    date = row["Date"].strftime("%Y-%m-%d")

    festival, ama, eka, pur, tithi, nak = get_festival(date)

    results.append({
        "Date": date,
        "Market": row["Market"],
        "Grade": row["Grade"],

        "Min Price (Rs./ Qtl)": row["Min Price (Rs./ Qtl)"],
        "Max Price (Rs./Qtl)": row["Max Price (Rs./Qtl)"],
        "Modal Price (Rs./Qtl)": row["Modal Price (Rs./Qtl)"],

        "Festival": festival,
        "is_amavasya": ama,
        "is_ekadashi": eka,
        "is_purnima": pur,
        "Tithi": tithi,
        "Nakshatra": nak,

        "Day": row["Day"],
        "Month": row["Month"],
        "Year": row["Year"],
        "Week": row["Week"]
    })

# =============================
# SAVE
# =============================
final_df = pd.DataFrame(results)
final_df.to_csv(OUTPUT_FILE, index=False)

print("🔥 FESTIVAL DATA READY!")

Processing Festival Data: 100%|██████████| 16337/16337 [07:32<00:00, 36.13it/s]

🔥 FESTIVAL DATA READY!


In [ ]:
import pandas as pd
import requests
from tqdm import tqdm
import os

# =============================
# CONFIG
# =============================
INPUT_FILE = "nashik_onion_festival.csv"
BATCH_SIZE = 1000
OUTPUT_FOLDER = "batches"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =============================
# LOAD
# =============================
df = pd.read_csv(INPUT_FILE)

# =============================
# NORMALIZE MARKET
# =============================
def normalize_market(name):
    return (
        str(name).upper()
        .replace(" (", "(")
        .replace(") ", ")")
        .strip()
    )

df["Market"] = df["Market"].apply(normalize_market)

# =============================
# COORDS
# =============================
coords = {
    "NASIK": (19.9975, 73.7898),
    "LASALGAON": (20.1420, 74.2390),
    "PIMPALGAON": (20.2000, 74.0000),
    "CHANDVAD": (20.3300, 74.2500),
    "MANMAD": (20.2500, 74.4500),
    "SINNER": (19.8500, 74.0000),
    "DEVALA": (20.4000, 74.5000),

    "LASALGAON(NIPHAD)": (20.1420, 74.2390),
    "PIMPALGAON BASWANT(SAYKHEDA)": (20.1300, 73.9000),
    "LASALGAON(VINCHUR)": (20.1500, 74.2000),

    "NAMPUR": (20.4700, 74.2000),
    "YEOLA": (20.0500, 74.5000),
    "UMRANE": (20.3000, 74.2000),
    "SATANA": (20.6000, 74.0000),
    "KALVAN": (20.5000, 74.0000),

    "DINDORI": (20.2000, 73.9000),
    "DINDORI(VANI)": (20.2000, 73.9000),
    "NANDGAON": (20.3000, 74.6500),
    "SURAGANA": (20.5500, 73.6500),

    "SHIVSIDDHA GOVIND PRODUCER COMPANY LIMITED SANCHAL": (20.0000, 74.0000),
    "MALHARSHREE FARMERS PRODUCER CO LTD": (20.1000, 74.2000),
    "SHREE RAMESHWAR KRUSHI MARKET": (20.1000, 74.1000),
    "MANKAMNESHWAR FARMAR PRODUCER COLTD SANCHALIT MANK": (20.1500, 74.1500)
}

coords = {normalize_market(k): v for k, v in coords.items()}

# =============================
# STRICT CHECK
# =============================
missing = set(df["Market"].unique()) - set(coords.keys())

if missing:
    print("❌ Missing coordinates:", missing)
    raise ValueError("Fix coords first")

# =============================
# WEATHER CACHE
# =============================
weather_cache = {}

def get_weather(lat, lon, date):

    key = f"{lat}_{lon}_{date}"

    if key in weather_cache:
        return weather_cache[key]

    try:
        url = "https://archive-api.open-meteo.com/v1/archive"

        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": date,
            "end_date": date,
            "daily": "temperature_2m_max,precipitation_sum",
            "timezone": "auto"
        }

        res = requests.get(url, params=params)
        data = res.json()

        temp = data["daily"]["temperature_2m_max"][0]
        rain = data["daily"]["precipitation_sum"][0]

    except Exception as e:
        print("❌ ERROR:", e)
        temp, rain = 30, 0

    heat = "Yes" if temp > 35 else "No"

    if rain > 20:
        rain_alert = "Heavy"
    elif rain > 5:
        rain_alert = "Moderate"
    else:
        rain_alert = "No"

    weather_cache[key] = (temp, rain, heat, rain_alert)

    return temp, rain, heat, rain_alert

# =============================
# BATCH PROCESSING 🔥
# =============================
num_batches = len(df) // BATCH_SIZE + 1

for batch_num in range(num_batches):

    start = batch_num * BATCH_SIZE
    end = start + BATCH_SIZE

    batch_df = df.iloc[start:end]

    if batch_df.empty:
        continue

    print(f"\n🚀 Processing Batch {batch_num+1}/{num_batches}")

    results = []

    for _, row in tqdm(batch_df.iterrows(), total=len(batch_df)):

        date = row["Date"]
        market = row["Market"]

        lat, lon = coords[market]

        temp, rain, heat, rain_alert = get_weather(lat, lon, date)

        if heat == "Yes" and rain < 2:
            advisory = "Irrigation needed | Heat stress risk"
        elif rain_alert == "Heavy":
            advisory = "Heavy rain expected"
        elif rain > 5:
            advisory = "Good moisture"
        else:
            advisory = "Normal conditions"

        row_dict = row.to_dict()

        row_dict.update({
            "Temp Max (°C)": temp,
            "Rain (mm)": rain,
            "Heat Stress": heat,
            "Rain Alert": rain_alert,
            "Agro Advisory": advisory
        })

        results.append(row_dict)

    batch_output = pd.DataFrame(results)
    batch_output.to_csv(f"{OUTPUT_FOLDER}/batch_{batch_num}.csv", index=False)

    print(f"✅ Saved batch_{batch_num}.csv")

print("\n🔥 ALL BATCHES DONE!")


🚀 Processing Batch 1/17


100%|██████████| 1000/1000 [10:37<00:00,  1.57it/s]


✅ Saved batch_0.csv

🚀 Processing Batch 2/17


100%|██████████| 1000/1000 [10:09<00:00,  1.64it/s]


✅ Saved batch_1.csv

🚀 Processing Batch 3/17


...

 44%|████▍     | 441/1000 [04:15<05:40,  1.64it/s]

❌ ERROR: 'daily'


 44%|████▍     | 442/1000 [04:15<05:37,  1.65it/s]

❌ ERROR: 'daily'


 44%|████▍     | 443/1000 [04:16<05:45,  1.61it/s]

In [3]:
import pandas as pd
import requests
from tqdm import tqdm
import os

# =============================
# CONFIG
# =============================
INPUT_FILE = "nashik_onion_festival.csv"
BATCH_SIZE = 1000
OUTPUT_FOLDER = "batches"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =============================
# LOAD
# =============================
df = pd.read_csv(INPUT_FILE)

# =============================
# NORMALIZE MARKET
# =============================
def normalize_market(name):
    return (
        str(name).upper()
        .replace(" (", "(")
        .replace(") ", ")")
        .strip()
    )

df["Market"] = df["Market"].apply(normalize_market)

# =============================
# COORDS
# =============================
coords = {
    "NASIK": (19.9975, 73.7898),
    "LASALGAON": (20.1420, 74.2390),
    "PIMPALGAON": (20.2000, 74.0000),
    "CHANDVAD": (20.3300, 74.2500),
    "MANMAD": (20.2500, 74.4500),
    "SINNER": (19.8500, 74.0000),
    "DEVALA": (20.4000, 74.5000),

    "LASALGAON(NIPHAD)": (20.1420, 74.2390),
    "PIMPALGAON BASWANT(SAYKHEDA)": (20.1300, 73.9000),
    "LASALGAON(VINCHUR)": (20.1500, 74.2000),

    "NAMPUR": (20.4700, 74.2000),
    "YEOLA": (20.0500, 74.5000),
    "UMRANE": (20.3000, 74.2000),
    "SATANA": (20.6000, 74.0000),
    "KALVAN": (20.5000, 74.0000),

    "DINDORI": (20.2000, 73.9000),
    "DINDORI(VANI)": (20.2000, 73.9000),
    "NANDGAON": (20.3000, 74.6500),
    "SURAGANA": (20.5500, 73.6500),

    "SHIVSIDDHA GOVIND PRODUCER COMPANY LIMITED SANCHAL": (20.0000, 74.0000),
    "MALHARSHREE FARMERS PRODUCER CO LTD": (20.1000, 74.2000),
    "SHREE RAMESHWAR KRUSHI MARKET": (20.1000, 74.1000),
    "MANKAMNESHWAR FARMAR PRODUCER COLTD SANCHALIT MANK": (20.1500, 74.1500)
}

coords = {normalize_market(k): v for k, v in coords.items()}

# =============================
# STRICT CHECK
# =============================
missing = set(df["Market"].unique()) - set(coords.keys())

if missing:
    print("❌ Missing coordinates:", missing)
    raise ValueError("Fix coords first")

# =============================
# WEATHER CACHE
# =============================
weather_cache = {}

def get_weather(lat, lon, date):

    key = f"{lat}_{lon}_{date}"

    if key in weather_cache:
        return weather_cache[key]

    try:
        url = "https://archive-api.open-meteo.com/v1/archive"

        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": date,
            "end_date": date,
            "daily": "temperature_2m_max,precipitation_sum",
            "timezone": "auto"
        }

        res = requests.get(url, params=params)
        data = res.json()

        temp = data["daily"]["temperature_2m_max"][0]
        rain = data["daily"]["precipitation_sum"][0]

    except Exception as e:
        print("❌ ERROR:", e)
        temp, rain = 30, 0

    heat = "Yes" if temp > 35 else "No"

    if rain > 20:
        rain_alert = "Heavy"
    elif rain > 5:
        rain_alert = "Moderate"
    else:
        rain_alert = "No"

    weather_cache[key] = (temp, rain, heat, rain_alert)

    return temp, rain, heat, rain_alert

# =============================
# BATCH PROCESSING 🔥
# =============================
num_batches = len(df) // BATCH_SIZE + 1

for batch_num in range(13, num_batches):

    start = batch_num * BATCH_SIZE
    end = start + BATCH_SIZE

    batch_df = df.iloc[start:end]

    if batch_df.empty:
        continue

    print(f"\n🚀 Processing Batch {batch_num+1}/{num_batches}")

    results = []

    for _, row in tqdm(batch_df.iterrows(), total=len(batch_df)):

        date = row["Date"]
        market = row["Market"]

        lat, lon = coords[market]

        temp, rain, heat, rain_alert = get_weather(lat, lon, date)

        if heat == "Yes" and rain < 2:
            advisory = "Irrigation needed | Heat stress risk"
        elif rain_alert == "Heavy":
            advisory = "Heavy rain expected"
        elif rain > 5:
            advisory = "Good moisture"
        else:
            advisory = "Normal conditions"

        row_dict = row.to_dict()

        row_dict.update({
            "Temp Max (°C)": temp,
            "Rain (mm)": rain,
            "Heat Stress": heat,
            "Rain Alert": rain_alert,
            "Agro Advisory": advisory
        })

        results.append(row_dict)

    batch_output = pd.DataFrame(results)
    batch_output.to_csv(f"{OUTPUT_FOLDER}/batch_{batch_num}.csv", index=False)

    print(f"✅ Saved batch_{batch_num}.csv")

print("\n🔥 ALL BATCHES DONE!")


🚀 Processing Batch 14/17


100%|██████████| 1000/1000 [09:21<00:00,  1.78it/s]


✅ Saved batch_13.csv

🚀 Processing Batch 15/17


100%|██████████| 1000/1000 [09:53<00:00,  1.69it/s]


✅ Saved batch_14.csv

🚀 Processing Batch 16/17


100%|██████████| 1000/1000 [09:36<00:00,  1.73it/s]


✅ Saved batch_15.csv

🚀 Processing Batch 17/17


100%|██████████| 337/337 [03:46<00:00,  1.49it/s]

✅ Saved batch_16.csv

🔥 ALL BATCHES DONE!


In [4]:
import pandas as pd
import glob

files = glob.glob("batches/*.csv")

df = pd.concat([pd.read_csv(f) for f in files])

df.to_csv("nashik_onion_full.csv", index=False)

print("🔥 FINAL MERGED FILE READY!")

🔥 FINAL MERGED FILE READY!


In [5]:
import pandas as pd

# =============================
# FILES
# =============================
INPUT_FILE = "nashik_onion_full.csv"
OUTPUT_FILE = "nashik_onion_filtered.csv"

# =============================
# LOAD
# =============================
df = pd.read_csv(INPUT_FILE)

# =============================
# NORMALIZE MARKET NAMES
# =============================
def normalize_market(name):
    return (
        str(name).upper()
        .replace(" (", "(")
        .replace(") ", ")")
        .replace("  ", " ")
        .strip()
    )

df["Market"] = df["Market"].apply(normalize_market)

# =============================
# VALID MARKETS
# =============================
valid_markets = [
    "LASALGAON",
    "NANDGAON",
    "PIMPALGAON",
    "CHANDVAD",
    "MANMAD",
    "SINNER",
    "LASALGAON(NIPHAD)",
    "PIMPALGAON BASWANT(SAYKHEDA)",
    "LASALGAON(VINCHUR)",
    "YEOLA",
    "UMRANE",
    "KALVAN"
]

# =============================
# FILTER
# =============================
df_filtered = df[df["Market"].isin(valid_markets)]

# =============================
# SORT (IMPORTANT FOR ML)
# =============================
df_filtered["Date"] = pd.to_datetime(df_filtered["Date"])
df_filtered = df_filtered.sort_values(["Market", "Grade", "Date"])

# =============================
# SAVE
# =============================
df_filtered.to_csv(OUTPUT_FILE, index=False)

# =============================
# INFO
# =============================
print("✅ Filtered dataset saved!")

print("\n📊 Markets kept:")
print(df_filtered["Market"].unique())

print("\n📊 Rows before:", len(df))
print("📊 Rows after:", len(df_filtered))

✅ Filtered dataset saved!

📊 Markets kept:
['CHANDVAD' 'KALVAN' 'LASALGAON' 'LASALGAON(NIPHAD)' 'LASALGAON(VINCHUR)'
 'MANMAD' 'NANDGAON' 'PIMPALGAON' 'PIMPALGAON BASWANT(SAYKHEDA)' 'SINNER'
 'UMRANE' 'YEOLA']

📊 Rows before: 16337
📊 Rows after: 11448


C:\Users\Sahil.Rajankar\AppData\Local\Temp\ipykernel_1464\1984857722.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Date"] = pd.to_datetime(df_filtered["Date"])
